In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
prefix = "../data/processed/"
matchups = pd.read_csv(prefix + 'TeamStatisticsFrom2010RegMatchups.csv')

In [3]:
matchups.columns

Index(['gameId', 'gameDate', 'season', 'home_teamId', 'home_teamName',
       'home_rest_days', 'home_b2b', 'win', 'home_teamScore_roll10',
       'home_fieldGoalsPercentage_roll10',
       'home_threePointersPercentage_roll10',
       'home_freeThrowsPercentage_roll10', 'home_reboundsOffensive_roll10',
       'home_reboundsDefensive_roll10', 'home_assists_roll10',
       'home_turnovers_roll10', 'home_steals_roll10', 'home_blocks_roll10',
       'home_plusMinusPoints_roll10', 'home_win_roll10', 'away_teamId',
       'away_teamName', 'away_rest_days', 'away_b2b', 'away_teamScore_roll10',
       'away_fieldGoalsPercentage_roll10',
       'away_threePointersPercentage_roll10',
       'away_freeThrowsPercentage_roll10', 'away_reboundsOffensive_roll10',
       'away_reboundsDefensive_roll10', 'away_assists_roll10',
       'away_turnovers_roll10', 'away_steals_roll10', 'away_blocks_roll10',
       'away_plusMinusPoints_roll10', 'away_win_roll10',
       'teamScore_roll10_diff', 'fieldGoalsP

In [5]:
matchups

,gameId,gameDate,season,home_teamId,home_teamName,home_rest_days,home_b2b,win,home_teamScore_roll10,home_fieldGoalsPercentage_roll10,...,reboundsOffensive_roll10_diff,reboundsDefensive_roll10_diff,assists_roll10_diff,turnovers_roll10_diff,steals_roll10_diff,blocks_roll10_diff,plusMinusPoints_roll10_diff,win_roll10_diff,rest_days_diff,b2b_diff
0,21000031,2010-10-30,2010,1610612737,Hawks,0.0,1,1,111.500000,0.519500,...,6.500000,13.500000,5.000000,4.500000,-5.000000,5.500000,38.000000,1.000000,-1.0,1
1,21000054,2010-11-03,2010,1610612737,Hawks,0.0,1,1,105.500000,0.474750,...,-0.500000,4.000000,6.250000,0.000000,0.750000,2.250000,17.750000,1.000000,0.0,0
2,21000090,2010-11-07,2010,1610612737,Hawks,1.0,0,0,104.833333,0.496500,...,-4.100000,4.033333,3.700000,-2.333333,-3.166667,1.700000,9.433333,0.600000,0.0,0
3,21000108,2010-11-10,2010,1610612737,Hawks,1.0,0,0,104.000000,0.499375,...,-1.000000,1.250000,6.625000,-0.500000,-1.000000,2.750000,5.250000,0.375000,1.0,-1
4,21000121,2010-11-12,2010,1610612737,Hawks,1.0,0,0,102.555556,0.498667,...,-0.819444,0.402778,-0.819444,-0.888889,-2.361111,0.694444,2.236111,0.041667,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19125,22501077,2026-03-28,2025,1610612766,Hornets,1.0,0,0,117.900000,0.468700,...,0.300000,0.500000,0.300000,1.700000,-2.000000,-2.100000,11.400000,0.100000,-1.0,0
19126,22501085,2026-03-29,2025,1610612766,Hornets,0.0,1,0,117.300000,0.463500,...,1.200000,-2.500000,1.400000,1.700000,1.000000,1.200000,5.600000,0.000000,-1.0,1
19127,22501114,2026-04-02,2025,1610612766,Hornets,1.0,0,1,118.700000,0.473400,...,-0.400000,3.300000,-0.800000,2.300000,-2.200000,-1.200000,9.800000,0.400000,0.0,0
19128,22501120,2026-04-03,2025,1610612766,Hornets,0.0,1,1,119.700000,0.479400,...,5.100000,2.600000,-8.700000,-0.200000,-0.100000,-0.700000,18.500000,0.400000,-1.0,1


In [4]:
features_diff = [c for c in matchups.columns if c.endswith('_diff')]
features_diff

['teamScore_roll10_diff',
 'fieldGoalsPercentage_roll10_diff',
 'threePointersPercentage_roll10_diff',
 'freeThrowsPercentage_roll10_diff',
 'reboundsOffensive_roll10_diff',
 'reboundsDefensive_roll10_diff',
 'assists_roll10_diff',
 'turnovers_roll10_diff',
 'steals_roll10_diff',
 'blocks_roll10_diff',
 'plusMinusPoints_roll10_diff',
 'win_roll10_diff',
 'rest_days_diff',
 'b2b_diff']

In [6]:
matchups["season"].dtype

dtype('int64')

# Train on 2010-2024, keep 2025 as the test dataset.
## Cross validate from 2015 t0 2024 using the rolling 5 years for training
### RandomForest

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [6]:
feature_importance = []
model_accuracy = {}

for i in range(2015, 2025):
    matchups_tt = matchups[matchups["season"].isin(range(i - 5, i))]
    matchups_val = matchups[matchups["season"] == i]
    X_tt = matchups_tt[features_diff]
    y_tt = matchups_tt['win']
    X_val = matchups_val[features_diff]
    y_val = matchups_val['win']
    rf = RandomForestClassifier(
    n_estimators = 500, # number of trees in ensemble
    max_depth = 10, # max_depth of each tree
    min_samples_leaf = 5, 
    max_features = 3, # default is round(sqrt(num_features)), which in this case is 1.
    bootstrap= True, # sampling with replacement
    max_samples = 500, # number of training samples selected with replacement to build tree
    random_state = 216 # for consistency
    )
    rf.fit(X_tt, y_tt)
    feature_importance.append(rf.feature_importances_)
    y_pred = rf.predict(X_val)
    model_accuracy[i] = accuracy_score(y_val, y_pred)


In [12]:
feature_importance_df = pd.DataFrame(
    feature_importance,
    index=range(2015, 2025),
    columns=features_diff
)
feature_importance_df.index.name = "season"

model_accuracy_df = pd.DataFrame.from_dict(
    model_accuracy, orient="index", columns=["accuracy"]
)
model_accuracy_df.index.name = "season"

results_df = model_accuracy_df.join(feature_importance_df)
results_df.index.name = None

In [13]:
results_df

,accuracy,teamScore_roll10_diff,fieldGoalsPercentage_roll10_diff,threePointersPercentage_roll10_diff,freeThrowsPercentage_roll10_diff,reboundsOffensive_roll10_diff,reboundsDefensive_roll10_diff,assists_roll10_diff,turnovers_roll10_diff,steals_roll10_diff,blocks_roll10_diff,plusMinusPoints_roll10_diff,win_roll10_diff,rest_days_diff,b2b_diff
2015,0.643089,0.080113,0.093040,0.075788,0.073983,0.068761,0.075699,0.066085,0.064260,0.063242,0.060937,0.154412,0.094942,0.017633,0.011107
2016,0.630081,0.083924,0.094697,0.079681,0.070764,0.063001,0.071804,0.072153,0.065199,0.063063,0.061525,0.150981,0.092825,0.018933,0.011450
2017,0.621138,0.083845,0.096387,0.080527,0.073480,0.066396,0.070772,0.073286,0.063336,0.061904,0.063212,0.152209,0.086718,0.018902,0.009026
2018,0.637398,0.086099,0.092209,0.075773,0.071280,0.066646,0.065693,0.075958,0.064243,0.063169,0.062945,0.153223,0.094730,0.017523,0.010508
2019,0.612637,0.084396,0.087188,0.080969,0.075494,0.064469,0.074486,0.073660,0.066902,0.064898,0.064612,0.151515,0.084478,0.017739,0.009195
2020,0.600926,0.086771,0.089565,0.077718,0.076030,0.064153,0.077533,0.074974,0.069904,0.062488,0.066249,0.143777,0.083680,0.017529,0.009631
2021,0.618699,0.088115,0.091817,0.080361,0.080953,0.068476,0.073759,0.074531,0.068140,0.061084,0.064931,0.141339,0.077298,0.018804,0.010392
2022,0.595122,0.090186,0.090681,0.078788,0.078380,0.067820,0.079592,0.071772,0.070304,0.065582,0.063511,0.138477,0.079945,0.016401,0.008562
2023,0.639024,0.083201,0.088706,0.083517,0.083381,0.068706,0.080589,0.078888,0.073542,0.064748,0.064404,0.130875,0.070651,0.018039,0.010754
2024,0.648980,0.083632,0.089014,0.082641,0.077467,0.074485,0.080465,0.075280,0.074526,0.062243,0.063608,0.133133,0.073492,0.018763,0.011250


### ExtraTreeClassifier

In [14]:
from sklearn.ensemble import ExtraTreesClassifier

feature_importance_et = []
model_accuracy_et = {}

for i in range(2015, 2025):
    matchups_tt = matchups[matchups["season"].isin(range(i - 5, i))]
    matchups_val = matchups[matchups["season"] == i]
    X_tt = matchups_tt[features_diff]
    y_tt = matchups_tt['win']
    X_val = matchups_val[features_diff]
    y_val = matchups_val['win']
    et = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=10,
        min_samples_leaf=5,
        max_features=3,
        bootstrap=False,  # Extra Trees default is no bootstrap
        random_state=216
    )
    et.fit(X_tt, y_tt)
    feature_importance_et.append(et.feature_importances_)
    y_pred = et.predict(X_val)
    model_accuracy_et[i] = accuracy_score(y_val, y_pred)

In [15]:
feature_importance_et_df = pd.DataFrame(
    feature_importance_et,
    index=range(2015, 2025),
    columns=features_diff
)
feature_importance_et_df.index.name = "season"

model_accuracy_et_df = pd.DataFrame.from_dict(
    model_accuracy_et, orient="index", columns=["accuracy"]
)
model_accuracy_et_df.index.name = "season"

results_et = model_accuracy_et_df.join(feature_importance_et_df)
results_et.index.name = None

results_et

,accuracy,teamScore_roll10_diff,fieldGoalsPercentage_roll10_diff,threePointersPercentage_roll10_diff,freeThrowsPercentage_roll10_diff,reboundsOffensive_roll10_diff,reboundsDefensive_roll10_diff,assists_roll10_diff,turnovers_roll10_diff,steals_roll10_diff,blocks_roll10_diff,plusMinusPoints_roll10_diff,win_roll10_diff,rest_days_diff,b2b_diff
2015,0.650407,0.065735,0.093543,0.030666,0.024854,0.025661,0.045721,0.033146,0.023587,0.030739,0.028595,0.287958,0.267180,0.016985,0.025631
2016,0.630894,0.089642,0.104519,0.039638,0.021587,0.021296,0.039794,0.038370,0.021492,0.026913,0.028185,0.263135,0.262066,0.017234,0.026129
2017,0.634959,0.095470,0.102252,0.040597,0.024611,0.023827,0.031664,0.049782,0.023439,0.025206,0.026388,0.269868,0.241027,0.017834,0.028035
2018,0.634959,0.097945,0.093967,0.039646,0.024124,0.021711,0.031367,0.052717,0.024124,0.023272,0.029060,0.260844,0.255095,0.016452,0.029676
2019,0.618132,0.100805,0.085966,0.039986,0.022495,0.019963,0.036582,0.055146,0.025071,0.025336,0.031011,0.275036,0.236631,0.017705,0.028268
2020,0.598148,0.094486,0.089882,0.036962,0.023293,0.021322,0.044013,0.048795,0.030933,0.024237,0.040339,0.260083,0.235759,0.018169,0.031728
2021,0.627642,0.091733,0.091133,0.043112,0.030884,0.023656,0.041690,0.038033,0.030116,0.026371,0.036967,0.253689,0.240333,0.017104,0.035179
2022,0.605691,0.093585,0.083924,0.036689,0.035510,0.024217,0.055173,0.035542,0.035370,0.027242,0.037140,0.243534,0.236737,0.020645,0.034693
2023,0.630894,0.073479,0.076900,0.047387,0.043008,0.025042,0.058430,0.045739,0.045506,0.027073,0.032274,0.250068,0.211362,0.023770,0.039962
2024,0.657143,0.075143,0.082022,0.052450,0.035394,0.028390,0.055604,0.031743,0.051147,0.026272,0.027477,0.243642,0.226706,0.021497,0.042514


### XGBoost

In [7]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

In [8]:
# Features and target
features_diff = [c for c in matchups.columns if c.endswith('_diff')]
X = matchups[features_diff]
y = matchups['win']

In [10]:
# Train/test split — keep temporal order, no shuffling
split_idx = int(len(matchups) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

In [11]:
# Model
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [12]:
# Evaluate
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Log Loss:  {log_loss(y_test, y_prob):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_prob):.4f}")

Accuracy:  0.6328
Log Loss:  0.6448
ROC AUC:   0.6744
